In [1]:
import torch
import requests
import json
import base64
import os
import warnings
warnings.filterwarnings("ignore")

# ====================== Core Configuration ======================
# 1. POE API Configuration
POE_API_KEY = "sk-poe-tgdCfdGh0feZtd1TdEU1LMPfeR96XY2HzAFHrHhjWdE"
POE_BASE_URL = "https://api.poe.com/v1"
POE_CHAT_COMPLETIONS_URL = f"{POE_BASE_URL}/chat/completions"
POE_MODEL = "gpt-4o-mini"

# 2. Sticker Style & Keywords
TARGET_STYLES = ["cartoon", "watercolor", "flat"]
STICKER_KEYWORDS = [
    "sticker", "high resolution", "minimal background",
    "clean edges", "vector style", "white background", "4k"
]

# 3. Batch Processing Configuration
IMAGE_FOLDER_PATH = "/kaggle/input/datasets/snnn9017/animal-images"  # Folder path for 20 original animal images
SUPPORTED_FORMATS = [".jpg", ".jpeg", ".png"]  # Supported image formats
JSON_SAVE_PATH = "./batch_sticker_prompts.json"  # Path to save batch prompts

# 4. SD Model Configuration (Keep basic config)
MODEL_ID = "sd-legacy/stable-diffusion-v1-5"
DEVICE = "cuda"
TORCH_DTYPE = torch.float16

# ====================== Tool Functions ======================
def save_prompt_to_json(prompt_data_list):
    """Save prompts to JSON in batch (overwrite/append mode)"""
    # Read existing data if file exists (append mode)
    if os.path.exists(JSON_SAVE_PATH):
        with open(JSON_SAVE_PATH, "r", encoding="utf-8") as f:
            try:
                existing_data = json.load(f)
                existing_data = existing_data if isinstance(existing_data, list) else []
            except json.JSONDecodeError:
                existing_data = []
    else:
        existing_data = []

    # Append newly generated prompt data
    existing_data.extend(prompt_data_list)

    # Save to JSON
    with open(JSON_SAVE_PATH, "w", encoding="utf-8") as f:
        json.dump(existing_data, f, indent=4, ensure_ascii=False)

    print(f"✅ Batch prompt save completed! Total {len(prompt_data_list)} entries. File path: {JSON_SAVE_PATH}")

def encode_image_to_base64(image_path):
    """Encode image to Base64 format"""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

def get_all_animal_images(folder_path):
    """Get paths of all valid animal images in the folder"""
    image_paths = []
    for file_name in os.listdir(folder_path):
        file_ext = os.path.splitext(file_name)[1].lower()
        if file_ext in SUPPORTED_FORMATS:
            full_path = os.path.join(folder_path, file_name)
            image_paths.append(full_path)
    return image_paths

# ====================== Core Function: Single Image → Prompt ======================
def generate_prompt_for_single_image(image_path):
    """Generate prompt for single image (multi-style)"""
    # Step 1: Encode image
    try:
        base64_image = encode_image_to_base64(image_path)
    except Exception as e:
        print(f"❌ Image encoding failed {image_path}: {e}")
        return None

    # Step 2: Build multi-style prompt instruction
    style_text = ", ".join(TARGET_STYLES)
    system_prompt = f"""
    You are a professional SD sticker prompt generator.
    Task:
    1. Analyze the input animal image in detail (species, color, posture, key features);
    2. Generate a high-quality sticker prompt with {style_text} mixed styles, containing:
       - Detailed animal features (from image analysis)
       - {style_text} style description (natural fusion)
       - Sticker core attributes: {', '.join(STICKER_KEYWORDS)}
    3. Output format (ONLY pure text, no special syntax):
       Raw Description: [simple animal description]
       SD Prompt: [complete mixed-style sticker prompt]
    """

    # Step 3: Call POE API
    headers = {
        "Authorization": f"Bearer {POE_API_KEY}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": POE_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}},
                    {"type": "text", "text": "Analyze this animal image and generate mixed-style sticker prompt"}
                ]
            }
        ],
        "max_tokens": 200,
        "temperature": 0.6
    }

    try:
        response = requests.post(POE_CHAT_COMPLETIONS_URL, headers=headers, json=payload, timeout=30)
        response.raise_for_status()
        result = response.json()
    except Exception as e:
        print(f"❌ API call failed {image_path}: {e}")
        return None

    # Step 4: Parse results
    content = result["choices"][0]["message"]["content"]
    try:
        raw_desc = content.split("Raw Description:")[1].split("SD Prompt:")[0].strip()
        sd_prompt = content.split("SD Prompt:")[1].strip()
    except IndexError:
        print(f"❌ Result parsing failed {image_path}, response content: {content}")
        return None

    # Step 5: Construct prompt data for single image
    prompt_data = {
        "file_name": os.path.basename(image_path),
        "raw_image_description": raw_desc,
        "sd_sticker_prompt": sd_prompt
    }

    print(f"✅ Generation completed {os.path.basename(image_path)}: {sd_prompt[:50]}...")
    return prompt_data

# ====================== Core Function: Batch Generate Prompts ======================
def batch_generate_prompts():
    """Batch generate prompts for 20 images"""
    # Step 1: Get all animal image paths
    image_paths = get_all_animal_images(IMAGE_FOLDER_PATH)
    if not image_paths:
        print(f"❌ No valid images found in folder {IMAGE_FOLDER_PATH}! Supported formats: {SUPPORTED_FORMATS}")
        return

    print(f"📁 Found {len(image_paths)} animal images, starting batch prompt generation...")

    # Step 2: Iterate to generate prompts for single images
    prompt_data_list = []
    for idx, image_path in enumerate(image_paths, 1):
        print(f"\n[{idx}/{len(image_paths)}] Processing image: {os.path.basename(image_path)}")
        prompt_data = generate_prompt_for_single_image(image_path)
        if prompt_data:
            prompt_data_list.append(prompt_data)

    # Step 3: Save to JSON in batch
    if prompt_data_list:
        save_prompt_to_json(prompt_data_list)
    else:
        print("❌ No valid prompts generated!")

# ====================== Run Batch Processing ======================
if __name__ == "__main__":
    # Check if folder exists
    if not os.path.exists(IMAGE_FOLDER_PATH):
        os.makedirs(IMAGE_FOLDER_PATH)
        print(f"⚠️ Folder {IMAGE_FOLDER_PATH} created. Please place original animal images in this folder and re-run!")
    else:
        # Execute batch generation
        batch_generate_prompts()

        # Optional: Verify generated JSON file
        if os.path.exists(JSON_SAVE_PATH):
            with open(JSON_SAVE_PATH, "r", encoding="utf-8") as f:
                data = json.load(f)
                print(f"\n🔍 JSON verification: Total {len(data)} prompts generated. Example:")
                print(json.dumps(data[0], indent=4, ensure_ascii=False)[:200] + "...")

📁 Found 37 animal images, starting batch prompt generation...

[1/37] Processing image: animal_2.jpeg
✅ Generation completed animal_2.jpeg: Two adorable black cats with shiny fur, one sittin...

[2/37] Processing image: animal_26.jpeg
✅ Generation completed animal_26.jpeg: Cartoon and watercolor mixed-style sticker of a fl...

[3/37] Processing image: animal_5.jpeg
✅ Generation completed animal_5.jpeg: A group of cute emperor penguins walking in the sn...

[4/37] Processing image: animal_6.jpeg
✅ Generation completed animal_6.jpeg: A cute, fluffy white rabbit with gray ears, cartoo...

[5/37] Processing image: animal_28.jpeg
✅ Generation completed animal_28.jpeg: A cute, fluffy light brown cat with long fur and a...

[6/37] Processing image: animal_29.jpeg
✅ Generation completed animal_29.jpeg: Create a high-resolution sticker of a gray tabby c...

[7/37] Processing image: animal_15.jpeg
✅ Generation completed animal_15.jpeg: A cute gray cat with fluffy striped fur and large ...

[8/37